# 2.2 — Document Loading & Chunking

Before we can search documents, we need to:
1. **Load** them (PDF, text, web page)
2. **Chunk** them into smaller pieces the model can process

## Why Chunk?
- LLMs have a context window limit
- Smaller chunks = more precise retrieval
- Too small = loses context | Too large = too much noise

In [ ]:
!pip install langchain langchain-community pypdf --quiet

## 1. Create a Sample Text File

In [ ]:
sample_text = """
Acme Corp Employee Handbook

Section 1: Working Hours
Standard working hours are 9am to 5pm, Monday to Friday.
Employees may request flexible working hours with manager approval.
Overtime must be pre-approved and will be compensated at 1.5x the hourly rate.

Section 2: Leave Policy
All full-time employees receive 20 days of annual leave per year.
Sick leave is up to 10 days per year with a medical certificate.
Parental leave is 16 weeks fully paid for primary caregivers.
Leave requests must be submitted at least 2 weeks in advance.

Section 3: Remote Work
Employees may work remotely up to 3 days per week.
A reliable internet connection is required for remote work.
Remote workers must be available during core hours: 10am to 3pm.
All remote work equipment is provided by the company.

Section 4: Code of Conduct
Employees are expected to treat colleagues with respect and professionalism.
Harassment, discrimination, or bullying of any kind will not be tolerated.
Confidential company information must not be shared outside the organization.
Violations of the code of conduct may result in disciplinary action.

Section 5: Benefits
Health insurance is provided for all full-time employees and their immediate family.
A gym membership subsidy of $50 per month is available.
Employees receive a $1,000 annual learning and development budget.
Free meals are provided in the office cafeteria.
"""

with open('employee_handbook.txt', 'w') as f:
    f.write(sample_text)

print('Sample file created: employee_handbook.txt')

## 2. Load the Document

In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader('employee_handbook.txt')
documents = loader.load()

print(f'Number of documents loaded: {len(documents)}')
print(f'Total characters: {len(documents[0].page_content)}')
print()
print('First 200 characters:')
print(documents[0].page_content[:200])

## 3. Chunking — RecursiveCharacterTextSplitter

This is the most commonly used splitter. It tries to split on paragraph breaks, then sentences, then words.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,      # max characters per chunk
    chunk_overlap=50,    # overlap between chunks to preserve context
)

chunks = splitter.split_documents(documents)

print(f'Number of chunks: {len(chunks)}')
print()
for i, chunk in enumerate(chunks):
    print(f'--- Chunk {i+1} ({len(chunk.page_content)} chars) ---')
    print(chunk.page_content)
    print()

## 4. Effect of Chunk Size

Let's compare different chunk sizes.

In [ ]:
for size in [100, 300, 800]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=20)
    chunks = splitter.split_documents(documents)
    print(f'chunk_size={size:4d}  →  {len(chunks):2d} chunks')

## 5. Loading a PDF (if you have one)

In [ ]:
# Uncomment and replace with your PDF path
# from langchain_community.document_loaders import PyPDFLoader
#
# loader = PyPDFLoader('your_document.pdf')
# pages = loader.load()
# print(f'Loaded {len(pages)} pages')
# for page in pages[:2]:
#     print(f'Page {page.metadata["page"]}: {len(page.page_content)} chars')

print('Uncomment the code above to load a PDF file.')

## Summary

| Concept | Value |
|---------|-------|
| `chunk_size` | Controls how large each chunk is (characters) |
| `chunk_overlap` | How much adjacent chunks share (preserves context at boundaries) |
| `TextLoader` | Load plain text files |
| `PyPDFLoader` | Load PDF files (one Document per page) |
| `RecursiveCharacterTextSplitter` | Smart splitter that respects paragraph/sentence boundaries |